## Using Local Modules in Remote Electrons

When running electrons on remote executors (e.g., SLURM, AWS Batch), your tasks may need to import helper modules that exist on your local machine but are not installed as pip packages on the remote execution environment. `DepsLocal` solves this by serializing your local Python modules and recreating them in the remote environment before the electron runs.

### Prerequisites

1. [Start](./covalent_cli.rst) the Covalent server.

2. Import `covalent` and `DepsLocal`.

In [ ]:
import covalent as ct
from covalent import DepsLocal

### What is DepsLocal?

`DepsLocal` is a dependency class that packages local Python module files (`.py`) and ensures they are available in the remote execution environment. Unlike `DepsPip`, which installs packages from PyPI or a requirements file, `DepsLocal` copies the raw source files directly.

| Feature | `DepsPip` | `DepsLocal` |
|---------|-----------|-------------|
| Source | PyPI packages or `requirements.txt` | Local `.py` files |
| Mechanism | Runs `pip install` in the remote environment | Writes source files to `/tmp/covalent_depslocal/` and prepends to `sys.path` |
| Use case | Public or private pip-installable packages | Custom helper modules not available as pip packages |

### Procedure

#### Creating a Helper Module

First, create a local Python module that your electron will import. For this example, we write a simple helper module to the current directory.

In [ ]:
with open('my_helper.py', 'w') as f:
    f.write('''
def greet(name):
    return f"Hello, {name}!"

def add(a, b):
    return a + b
''')

#### Passing a Single Module Name

The simplest way to use `deps_local` is to pass a single module name (without the `.py` extension) as a string.

In [ ]:
@ct.electron(
    deps_local="my_helper"
)
def remote_greet(name):
    from my_helper import greet
    return greet(name)

#### Passing Multiple Module Names

You can also pass a list of module names to include multiple local modules.

In [ ]:
with open('math_utils.py', 'w') as f:
    f.write('''
def square(x):
    return x * x

def cube(x):
    return x * x * x
''')

In [ ]:
@ct.electron(
    deps_local=["my_helper", "math_utils"]
)
def remote_compute(name, x):
    from my_helper import greet, add
    from math_utils import square
    greeting = greet(name)
    result = square(x)
    return f"{greeting} The square of {x} is {result}."

#### Using the DepsLocal Object Explicitly

For more control, you can create a `DepsLocal` object explicitly. This allows you to specify a custom `path` to search for modules (defaults to the current working directory).

In [ ]:
deps = DepsLocal(
    modules=["my_helper", "math_utils"],
    path="."
)

@ct.electron(
    deps_local=deps
)
def remote_compute_explicit(name, x):
    from my_helper import greet
    from math_utils import cube
    greeting = greet(name)
    result = cube(x)
    return f"{greeting} The cube of {x} is {result}."

#### How It Works

When the electron is dispatched to a remote executor:

1. Covalent reads the specified `.py` files from the local `path`.
2. The module files are serialized and sent to the remote execution environment.
3. Before the electron function runs, the modules are written to `/tmp/covalent_depslocal/`.
4. That directory is prepended to `sys.path`, so `import` statements find the local modules.
5. If a module file already exists in the remote environment with identical content, it is not overwritten.

#### Limitations

- **Flat directory only**: `DepsLocal` does not support nested package directories. Each module must be a single `.py` file.
- **Hardcoded destination**: Modules are always written to `/tmp/covalent_depslocal/` in the remote environment. This path is not configurable.
- **No package structure**: Because modules are placed in a flat directory, you cannot use relative imports between local modules as you would in a proper Python package.
- **Source files only**: `DepsLocal` copies `.py` source files. It does not compile or install packages. For compiled extensions or complex packages, use `DepsPip` instead.

### Complete the Workflow

Regardless of how you specify local modules, construct and dispatch the workflow as usual.

In [ ]:
@ct.lattice
def workflow(name, x):
    return remote_compute(name=name, x=x)

dispatch_id = ct.dispatch(workflow)(name="World", x=5)
res = ct.get_result(dispatch_id, wait=True)
print(res.result)

### See Also

[Adding Pip Dependencies to an Electron](../coding/add_pip_dependencies_to_electron.ipynb)

[Adding Bash Dependencies to an Electron](../coding/add_bash_dependencies_to_electron.ipynb)

[Adding Callable Function Dependencies to an Electron](../coding/add_callable_dependencies_to_electron.ipynb)